# Improved TED Views Prediction Model

This notebook improves the original model by:

- Using transcript text as an additional feature
- Better text preprocessing
- Feature engineering
- TF-IDF + SVD dimensionality reduction
- Ensemble learning (`Ridge` + `HistGradientBoostingRegressor`)
- Cross-validation evaluation
- More robust preprocessing pipeline


In [48]:
import pandas as pd
import numpy as np

from sklearn.model_selection import KFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor, VotingRegressor

import warnings
warnings.filterwarnings("ignore")


## Load Data

In [49]:
test = pd.read_csv(r"C:\Users\hp\Desktop\Data_analysis-and-ML\test (2).csv")
train = pd.read_csv(r"C:\Users\hp\Desktop\Data_analysis-and-ML\train (2).csv")
transcripts = pd.read_csv(r"C:\Users\hp\Desktop\Data_analysis-and-ML\transcripts.csv")

print(train.shape)
print(test.shape)
print(transcripts.shape)


(2040, 18)
(510, 17)
(2467, 2)


## Merge Transcript Data

In [50]:
possible_id_cols = ["transcript", "transcript_id", "id"]

merge_col = None
for col in possible_id_cols:
    if col in train.columns and col in transcripts.columns:
        merge_col = col
        break

if merge_col and "transcript" in transcripts.columns:
    train = train.merge(
        transcripts[[merge_col, "transcript"]],
        on=merge_col,
        how="left"
    )

    test = test.merge(
        transcripts[[merge_col, "transcript"]],
        on=merge_col,
        how="left"
    )

print("Merge column:", merge_col)


Merge column: None


## Feature Engineering

In [51]:
text_cols = [
    "title",
    "description",
    "main_speaker",
    "speaker_occupation",
    "event",
    "tags",
    "ratings",
    "transcript"
]

for col in text_cols:
    if col not in train.columns:
        train[col] = ""
    if col not in test.columns:
        test[col] = ""

train[text_cols] = train[text_cols].fillna("")
test[text_cols] = test[text_cols].fillna("")

num_cols = [
    "duration",
    "languages",
    "num_speaker",
    "comments"
]

for col in num_cols:
    if col not in train.columns:
        train[col] = np.nan
    if col not in test.columns:
        test[col] = np.nan

for df in [train, test]:
    df["title_len"] = df["title"].str.len()
    df["desc_len"] = df["description"].str.len()
    df["transcript_len"] = df["transcript"].str.len()
    df["num_tags"] = df["tags"].astype(str).apply(lambda x: len(x.split(",")))

extra_num_cols = [
    "title_len",
    "desc_len",
    "transcript_len",
    "num_tags"
]

num_cols += extra_num_cols

train["all_text"] = train[text_cols].agg(" ".join, axis=1)
test["all_text"] = test[text_cols].agg(" ".join, axis=1)

print("Feature engineering completed.")


Feature engineering completed.


## Target Transformation

In [52]:
# Convert views to numeric safely
train["views"] = pd.to_numeric(train["views"], errors="coerce")

# Drop rows where views are NaN or <= 0
train = train.dropna(subset=["views"])
train = train[train["views"] > 0]

# Separate target and features after dropping the bad rows
y = np.log1p(train["views"]) 
train = train.drop(columns=["views"])

## Preprocessing

In [53]:
text_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=40000,
        ngram_range=(1, 2),
        stop_words="english",
        sublinear_tf=True
    )),
    ("svd", TruncatedSVD(n_components=300, random_state=42))
])

num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

preprocessor = ColumnTransformer([
    ("text", text_pipeline, "all_text"),
    ("num", num_pipeline, num_cols)
])


## Train Ensemble Model

In [54]:
ridge_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", Ridge(alpha=5.0))
])

gbr_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", HistGradientBoostingRegressor(
        max_iter=300,
        learning_rate=0.05,
        max_depth=6,
        random_state=42
    ))
])

ensemble = VotingRegressor([
    ("ridge", ridge_model),
    ("gbr", gbr_model)
])

ensemble.fit(train, y)

print("Training completed.")


Training completed.


## Cross Validation

In [55]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(
    ensemble,
    train,
    y,
    scoring="neg_root_mean_squared_error",
    cv=kf,
    n_jobs=-1
)

rmse_scores = -scores

print("Fold RMSE scores:")
print(rmse_scores)

print("\nMean RMSE:", rmse_scores.mean())


Fold RMSE scores:
[0.61201328 0.68698983 0.66916842 0.66019311 0.55590722]

Mean RMSE: 0.6368543719446571


## Predict

In [56]:
test_pred_log = ensemble.predict(test)

test_pred = np.expm1(test_pred_log)
test_pred = np.clip(test_pred, 0, None)

print(test_pred[:5])


[ 713913.01881233  531217.60927504 1069293.67548085  352771.62250171
 1430352.87244564]


## Submission File

In [57]:
submission = pd.DataFrame({
    "id": test["id"],
    "views": test_pred
})

submission.to_csv("improved_submission.csv", index=False)

print(submission.head())
print("\nSaved as improved_submission.csv")


     id         views
0    56  7.139130e+05
1   194  5.312176e+05
2  2225  1.069294e+06
3   233  3.527716e+05
4  1902  1.430353e+06

Saved as improved_submission.csv
